In [ ]:
import torch
import numpy as np
import pandas as pd
import torch.nn as nn

from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, Dataset, DataLoader

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

from google.colab import data_table
data_table.enable_dataframe_formatter()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Importing pandas dataframe and aligning it according to the problem.

In [ ]:
columns = ['movie_id', 'movie_title', 'release_date', 'video_release_date', 'IMDb_URL'] + [f'genre_{i}' for i in range(19)]

movie_details = pd.read_csv(
    "/content/drive/MyDrive/ml-100k/u.item",
    sep='|',
    encoding='latin-1',
    names=columns,
    header=None
)
movie_details.head()

,movie_id,movie_title,release_date,video_release_date,IMDb_URL,genre_0,genre_1,genre_2,genre_3,genre_4,genre_5,genre_6,genre_7,genre_8,genre_9,genre_10,genre_11,genre_12,genre_13,genre_14,genre_15,genre_16,genre_17,genre_18
0,1,Toy Story (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0
1,2,GoldenEye (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?GoldenEye%20(...,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
2,3,Four Rooms (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Four%20Rooms%...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
3,4,Get Shorty (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Get%20Shorty%...,0,1,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0
4,5,Copycat (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Copycat%20(1995),0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0


In [ ]:
data = pd.read_csv(
    filepath_or_buffer='/content/drive/MyDrive/ml-100k/u.data',
    sep = '\t',
    names = ['user_id', 'movie_id', 'rating', 'timestamp']
)

data = data.drop('timestamp', axis = 1)

data.shape

data.iloc[5]

data['user_id'] -= 1
data['movie_id'] -= 1

data

,user_id,movie_id,rating
0,195,241,3
1,185,301,3
2,21,376,1
3,243,50,2
4,165,345,1
...,...,...,...
99995,879,475,3
99996,715,203,5
99997,275,1089,1
99998,12,224,2


In [ ]:
data['rating'].mean()

np.float64(3.52986)

# Splitting the data between train and test

In [ ]:
train, test = train_test_split(data, test_size = .2, shuffle = True, random_state = 42)

# Creating a custom dataset to fetch the data columns as needed & also creating dataloaders


In [ ]:
class MovieLensDataset(Dataset):

    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]


        user_id = torch.tensor(row['user_id'], dtype = torch.long)
        movie_id = torch.tensor(row['movie_id'], dtype = torch.long)
        ratings = torch.tensor(row['rating'], dtype = torch.float32)

        return user_id, movie_id, ratings


train_dataset = MovieLensDataset(train)
test_dataset = MovieLensDataset(test)

trainloader = DataLoader(train_dataset, batch_size=128, shuffle=True)
testloader = DataLoader(test_dataset, batch_size = len(test_dataset))


user_id, movie_id, ratings = next(iter(trainloader))

print(user_id.shape, user_id.dtype)
print(movie_id.shape, movie_id.dtype)
print(ratings.shape, ratings.dtype)

torch.Size([1024]) torch.int64
torch.Size([1024]) torch.int64
torch.Size([1024]) torch.float32


In [ ]:
num_movies = len(pd.unique(data['movie_id']))
num_users = len(pd.unique(data['user_id']))

# Model Class Creation

In [ ]:
def createTheModel(num_users, num_movies, emb_size):
    class MatrixFactorization(nn.Module):
        def __init__(self, num_user, num_movies, num_embeddings):
            super().__init__()

            self.user_embeddings = nn.Embedding(
                num_user,
                embedding_dim = num_embeddings
            )

            self.movie_embeddings = nn.Embedding(
                num_movies,
                embedding_dim = num_embeddings
            )

            self.user_bias = nn.Embedding(
                num_user,
                1
            )

            self.movie_bias = nn.Embedding(
                num_movies,
                1
            )

        def forward(self, user_id, movie_id):

            user_vec = self.user_embeddings(user_id)
            movie_vector = self.movie_embeddings(movie_id)

            user_bias = self.user_bias(user_id)
            movie_bias = self.movie_bias(movie_id)

            user_bias = user_bias.squeeze()
            movie_bias = movie_bias.squeeze()

            dot_product = (user_vec * movie_vector).sum(dim = 1)

            predictions = (
                dot_product
                + user_bias
                + movie_bias
            )

            return predictions

    model = MatrixFactorization(num_user = num_users, num_movies = num_movies, num_embeddings = emb_size)

    lossfun = nn.MSELoss()

    optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)

    return model, lossfun, optimizer

# Training the Model

In [ ]:
def trainTheModel(model, lossfun, optimizer):

    torch.manual_seed(42)

    epochs = 30
    trainloss = torch.zeros(epochs)
    testloss = torch.zeros(epochs)
    trainRmse = torch.zeros(epochs)
    testRmse = torch.zeros(epochs)

    model.to(device)

    for epochi in range(epochs):

        model.train()

        batchLoss = []
        for user_id, movie_id, ratings in trainloader:


            user_id = user_id.to(device)
            movie_id = movie_id.to(device)
            ratings = ratings.to(device)

            prediction = model(
                user_id,
                movie_id,
            )

            loss = lossfun(prediction, ratings)
            batchLoss.append(loss.detach())

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        trainloss[epochi] = torch.mean(torch.stack(batchLoss))
        trainRmse[epochi] = torch.sqrt(trainloss[epochi])


        # Using test data to test the model
        Tuserid, Tmovieid, Tratings = next(iter(testloader))
        Tuserid, Tmovieid, Tratings = Tuserid.to(device), Tmovieid.to(device), Tratings.to(device)

        model.eval()

        with torch.no_grad():

            Tprediction = model(
                Tuserid,
                Tmovieid,
            )

        loss = lossfun(Tprediction,Tratings)
        testloss[epochi] = loss.item()
        testRmse[epochi] = torch.sqrt(loss)


        if epochi == epochs - 1:
            print(f"Final Results -> Train RMSE:{trainRmse[epochi]:.4f} & Test RMSE:{testRmse[epochi]:.4f}")
            print(f"Final Results -> Train loss:{trainloss[epochi]:.4f} & Test loss:{loss.item():.4f}", end = '\n\n')

    return trainloss, testloss, trainRmse, testRmse


# Perfomring experiment of embedding sizes


In [ ]:
emb_size = [8,16,32,64]
exp_results = []

for i, emb in enumerate(emb_size):
    torch.manual_seed(42)
    model, lossfun, optimizer = createTheModel(943, 1682, emb)
    trainloss, testloss, trainRmse, testRmse = trainTheModel(model, lossfun, optimizer)
    exp_results.append({
        'embedding' : emb,
        'TrainRMSE' : trainRmse[29],
        'TestRMSE'  : testRmse[29],
        'TrainLOSS' : trainloss[29],
        'TestLOSS'  : testloss[29]
    })

exp_results = pd.DataFrame(exp_results)
exp_results


# print(model.user_embeddings.weight.shape)
# print(model.movie_embeddings.weight.shape)
# model.movie_embeddings.weight[:5]

Final Results -> Train RMSE:2.1558 & Test RMSE:2.3323
Final Results -> Train loss:4.6474 & Test loss:5.4397

Final Results -> Train RMSE:2.5611 & Test RMSE:2.8974
Final Results -> Train loss:6.5595 & Test loss:8.3951

Final Results -> Train RMSE:3.0301 & Test RMSE:3.7837
Final Results -> Train loss:9.1815 & Test loss:14.3163

Final Results -> Train RMSE:3.9027 & Test RMSE:5.7264
Final Results -> Train loss:15.2308 & Test loss:32.7913



,embedding,TrainRMSE,TestRMSE,TrainLOSS,TestLOSS
0,8,tensor(2.1558),tensor(2.3323),tensor(4.6474),tensor(5.4397)
1,16,tensor(2.5611),tensor(2.8974),tensor(6.5595),tensor(8.3951)
2,32,tensor(3.0301),tensor(3.7837),tensor(9.1815),tensor(14.3163)
3,64,tensor(3.9027),tensor(5.7264),tensor(15.2308),tensor(32.7913)


In [ ]:
# first_15 = np.round(Tprediction.cpu().numpy()[:15], decimals = 0)
# print(first_15)
# print(Tratings.cpu().numpy()[:15])

# print(Tprediction.max())
# print(Tprediction.min())

In [ ]:
# def recommend_movies(model, user_id, top_k = 10):
#     score = model.movie_embeddings.weight @ model.user_embeddings.weight[user_id]
#     score += model.user_bias.weight[user_id]
#     score += model.movie_bias.weight.squeeze()

#     watched_movies = data[data["user_id"] == user_id]["movie_id"].values

#     score[watched_movies] = -float('inf')

#     values, idx = torch.topk(score, k = top_k)

#     return idx, values
